# Face Shape Classifier — Training Notebook

This notebook takes the `face_shape_training_data.csv` file (built earlier by measuring face photos with MediaPipe) and trains a machine learning model that can look at a face's ratios and guess its shape: Heart, Oblong, Oval, Round, or Square.

**The columns in the CSV:**
- `hr` — height ratio (face height ÷ cheekbone width)
- `fr` — forehead ratio (forehead width ÷ cheekbone width)
- `jr` — jaw ratio (jaw width ÷ cheekbone width)
- `label` — the correct face shape for that row

We'll go step by step: load the data, split it, train a model, check how good it is, then save it so it can be used back in the main camera app.

> **ARCHIVED** — superseded by the MobileNetV2 CNN trained via `train_face_shape_cnn_v2.ipynb`, which gets materially better accuracy than this ratio-based Random Forest. Kept for reference only.

## Step 1: Upload the CSV

Running the cell below will pop up a file picker. Choose `face_shape_training_data.csv` from your computer. Colab doesn't have access to your files until you upload them — this only needs to be done once per session.

In [ ]:
from google.colab import files

uploaded = files.upload()

## Step 2: Load and inspect the data

Now we read the CSV into a pandas DataFrame (basically a spreadsheet Python can work with). Then we print:
- The first 5 rows, so we can visually check the columns and values look right.
- A count of how many rows belong to each face shape, so we can confirm the dataset is balanced (roughly equal photos per shape) before training. A model trained on an unbalanced dataset tends to just guess the most common class.

In [ ]:
import pandas as pd

df = pd.read_csv('face_shape_training_data.csv')

print(df.head())
print()
print("Rows per face shape:")
print(df['label'].value_counts())

## Step 3: Split into training and testing sets

We can't test the model on the same data it learned from — that would be like grading a student using the exact questions they memorized the answers to. So we hold back 20% of the rows as a "test set" the model never sees during training, and use the other 80% to teach it.

`stratify=y` just makes sure both the training set and the test set keep the same mix of face shapes (e.g. not accidentally putting all the Round faces in the test set).

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['hr', 'fr', 'jr']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

## Step 4: Train a Random Forest classifier

A **Random Forest** is a collection of many decision trees (here, 100 of them). Each decision tree looks at the three ratios and makes a series of yes/no splits (e.g. "is `hr` greater than 1.3?") to guess a face shape. Any single tree can be wrong or overfit to quirks in the data, so the forest trains many slightly different trees (each on a random subset of the data) and then has them vote — the majority answer wins. This voting makes the overall prediction more accurate and less easily thrown off by noisy or unusual rows than one tree alone.

`random_state=42` just makes the randomness reproducible, so re-running this cell gives the same result every time.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained.")

## Step 5: Check how good the model is

Now we ask the trained model to predict the face shape for every row in the test set (the 20% it has never seen), and compare its guesses to the real labels.

- **Accuracy** — the percentage of test rows it got exactly right.
- **Classification report** — breaks accuracy down per face shape:
  - *Precision*: of everything the model called "Heart", what fraction actually was Heart?
  - *Recall*: of everything that actually was Heart, what fraction did the model catch?
- **Confusion matrix** — a table where rows are the true shape and columns are the predicted shape. The diagonal is correct guesses; anything off the diagonal shows exactly which shapes get mixed up with each other (e.g. Oval mistaken for Round).

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print("Confusion Matrix (rows = actual shape, columns = predicted shape):")
print(cm_df)

## Step 6: Save the trained model

`joblib.dump()` saves the trained model (all 100 trees and everything it learned) into a single file, `face_shape_model.joblib`. This means we don't have to retrain it every time — the main camera app can just load this file and start predicting face shapes immediately.

In [ ]:
import joblib

joblib.dump(model, 'face_shape_model.joblib')
print("Model saved as face_shape_model.joblib")

## Step 7: Download the model to your computer

Colab's files are temporary and get deleted when the session ends, so the last step is to download `face_shape_model.joblib` to your own machine. From there, drop it into the project folder so `main.py` can load it.

In [ ]:
from google.colab import files

files.download('face_shape_model.joblib')